Lucas Auada Braga
Vitor Kenzo Koga Onoue

4/5 - Vitor Onoue - Starting data preparation

4/6 - Vitor Onoue - Finished data preparation + EDA

5/31 - Lucas Braga - Second notebook containing model training steps

5/31 - Vitor Onoue - Merging both notebooks + adding comments

In this notebook, we are preparing and cleaning an e-commerce dataset from Olist, in order to train a model to predict if an order's delivery will be delayed.

After selecting the fields that will be used, we are also making a very simple EDA.

In [ ]:
import pandas as pd

Taking a look at each csv file

In [ ]:
orders = pd.read_csv('csvs/olist_orders_dataset.csv')
orders.sample()

In [ ]:
customers = pd.read_csv('csvs/olist_customers_dataset.csv')
customers.sample()

In [ ]:
order_items = pd.read_csv('csvs/olist_order_items_dataset.csv')
order_items.sample()

Treating orders

In [ ]:
# cleaning rows where the order_delivered_customer_date is null (treating as not delivered)
orders = orders.dropna(subset=['order_delivered_customer_date'])

# converting date columns to datetime format
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# creating a new column for the estimated delivery in days
orders['estimated_delivery_days'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days

# creating a new column for month of purchase
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.month

# creating a delay column by checking if the date the customer received the order is greater than what was expected
orders['delay'] = (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']).astype(bool)

Treating customers

In [ ]:
# for the customers csv, only using the customer_state field - removing rows with null values for this field
customers = customers.dropna(subset=['customer_state'])

Treating order_items

In [ ]:
# for the order_items csv, only using the freight_value field - removing rows with null values for this field
order_items = order_items.dropna(subset=['freight_value'])

Each row in order_items contains a product and an order it is associated to

Which means that we may have multiple rows with the same order_id, as an order can have multiple products

The goal is to predict a possible delay in the delivery of an **order**, so we are transforming multiple rows with the same order_id into a single row

We are considering each product having its own freight tax, and if an order contains, for example, 3 products, the total freight value will be the sum of all products' freight values

In [ ]:
order_items_agg = order_items.groupby('order_id').agg({
    'freight_value': 'sum',
    'price': 'sum',
    'order_item_id': 'count'
}).rename(columns={
    'price': 'order_price',
    'order_item_id': 'item_count'
}).reset_index()

Merging all the csvs together into a single csv, with only the fields we are going to use

In [ ]:
dataset = orders.merge(order_items_agg, on='order_id', how='left')
dataset = dataset.merge(customers, on='customer_id', how='left')

columns = ['order_purchase_timestamp',
                      'order_estimated_delivery_date', 'order_delivered_customer_date',
                      'customer_state', 'order_price', 'freight_value', 'estimated_delivery_days', 'purchase_month', 'delay']
dataset[columns].to_csv('csvs/clean_dataset.csv', index=False)

dataset[columns].sample()

Starting Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

dataset = pd.read_csv('csvs/clean_dataset.csv')

In [ ]:
# First five rows
dataset.head()

In [ ]:
# General info
print(dataset.info())

In [ ]:
# Checking null values - as they were removed previously, expected to be 0
print(dataset.isnull().sum())

In [ ]:
# Calculating delay
dataset['order_purchase_timestamp'] = pd.to_datetime(dataset['order_purchase_timestamp'])
dataset['order_estimated_delivery_date'] = pd.to_datetime(dataset['order_estimated_delivery_date'])
dataset['order_delivered_customer_date'] = pd.to_datetime(dataset['order_delivered_customer_date'])

dataset['delay_in_days'] = (dataset['order_delivered_customer_date'] - dataset['order_estimated_delivery_date']).dt.days
dataset['delay_in_days'].sample(5)

In [ ]:
# Delay histogram
sns.histplot(dataset['delay_in_days'].dropna(), bins=30, kde=True)
plt.title('Delay in Days Distribution')
plt.xlim(-100, 100)
plt.show()

In [ ]:
# Price vs Freight value scatter
sns.scatterplot(data=dataset, x='order_price', y='freight_value')
plt.title('Price vs Freight Value')
plt.xlabel('Price')
plt.ylabel('Freight Value')
plt.show()

Model training steps:

+ select the feature columns and the target column
+ split training and testing data
+ choose the algorithm and train the model
+ print and analyze the results

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

orders = pd.read_csv('csvs/clean_dataset.csv')


In [ ]:
# one-hot encoding customer_state
orders = pd.get_dummies(orders, columns=['customer_state'], drop_first=True)

# selecting features and target
x = orders[['order_price', 'freight_value', 'estimated_delivery_days', 'purchase_month'] + [col for col in orders.columns if col.startswith('customer_state_')]]
y = orders['delay']

# splitting the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

# normalizing the features, as logistic regression will be sensitive to the high/low values of order_price and freight_value
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
# training with logistic regression - the low amount of falses requires class balance
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(x_train_scaled, y_train)
y_pred_lr = lr.predict(x_test_scaled)

In [ ]:
print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
# training with random forest - the low amount of falses requires class balance
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)

In [ ]:
print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))